# 損失、優化器與正則化

對應原作業：Day 71–84。這是現代 PyTorch 練習；原始作業保留在 `../../homework/`。

從上到下執行即可。預設使用 CPU、合成資料，不需下載外部資料；改為真實資料時，先決定資料粒度與切分方式。

## 選法
多類互斥分類用 `CrossEntropyLoss`；二元／多標籤用 `BCEWithLogitsLoss`；回歸從 `MSELoss`／`L1Loss` 開始。`AdamW` 的 `weight_decay`、Dropout、BatchNorm 各處理不同問題，需在同一驗證方式下比較。

In [ ]:
import torch
from torch import nn

torch.manual_seed(42)
X = torch.randn(32, 10)
y = (X[:, 0] > 0).long()

def make_model():
    return nn.Sequential(nn.Linear(10, 24), nn.BatchNorm1d(24), nn.ReLU(),
                         nn.Dropout(p=0.2), nn.Linear(24, 2))

for name, optimizer_factory in [
    ("SGD+momentum", lambda m: torch.optim.SGD(m.parameters(), lr=0.02, momentum=0.9, nesterov=True)),
    ("AdamW", lambda m: torch.optim.AdamW(m.parameters(), lr=0.005, weight_decay=0.01)),
]:
    model = make_model()
    optimizer = optimizer_factory(model)
    model.train()
    optimizer.zero_grad(set_to_none=True)
    logits = model(X)
    loss = nn.CrossEntropyLoss()(logits, y)
    loss.backward()
    optimizer.step()
    print(name, "one-step loss:", loss.item())


In [ ]:
# L1 是額外加到 loss 的懲罰；要把量級與驗證分數一起比較。
model = make_model()
model.train()
logits = model(X)
cross_entropy = nn.CrossEntropyLoss()(logits, y)
l1_penalty = sum(parameter.abs().sum() for parameter in model.parameters())
total_loss = cross_entropy + 1e-5 * l1_penalty
total_loss.backward()
print("CE:", cross_entropy.item(), "CE + L1:", total_loss.item())
